# $\phi$ \ $\chi$ spectra
Comparing observed spectra of acoustic phase, $\phi$ and log-amplitude, $\chi$ with modelled spectra given Garrett-Munk internal waves spectra (calculated with WFT).

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from dotenv import find_dotenv, load_dotenv
from scipy import signal, optimize

from nescafe23_fluctuations import tools
import warnings
_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

## open observations

In [ ]:
m_data, F_phi_data, F_chi_data, F_phi_lm, F_phi_ls, F_chi_lm, F_chi_ls = tools.estimate_F_phi_chi()

### estimate observational noise

In [ ]:
F_phi_lm[ray_id].shape

In [ ]:
F_phi_obs_noise = {}
F_chi_obs_noise = {}

for ray_id in ray_ids:
    F_phi_obs_noise[ray_id] = 10**np.mean(np.log10(F_phi_lm[ray_id][-8:]))
    F_chi_obs_noise[ray_id] = 10**np.mean(np.log10(F_chi_lm[ray_id][-8:]))

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(6,4))
axes = axes.flatten()

for k, ray_id in enumerate(ray_ids):
    plt.sca(axes[k])

    # F_phi observed
    plt.plot(m_data, F_phi_lm[ray_id], c="C1", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_phi_lm[ray_id] * F_phi_ls[ray_id],
        F_phi_lm[ray_id] / F_phi_ls[ray_id],
        alpha=0.3,
        color="C1",
        lw=0,
    )

    # F_chi observed
    plt.plot(m_data, F_chi_lm[ray_id], c="C3", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_chi_lm[ray_id] * F_chi_ls[ray_id],
        F_chi_lm[ray_id] / F_chi_ls[ray_id],
        alpha=0.3,
        color="C3",
        lw=0,
    )

    plt.plot(m_data[-8:], F_phi_lm[ray_id][-8:])
    plt.hlines([F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id]], m_data[0], m_data[-1], colors=['C0', 'C2'])
    
    plt.yscale('log')
    plt.xscale('log')

    plt.title(ray_id)

    plt.xlabel("vertical wave number [1/m]")
    plt.ylabel("$F_{\\phi, \\chi}$ [$\\mathrm{rad} \\ m$]")

plt.tight_layout(pad=1)

## Compute internal wave spectra (WFT)

In [ ]:
m, F_phis, F_chis = tools.compute_F_phi_chi_n23(1)

## Build LS fit for spectra + noise model

In [ ]:
results = tools.fit_F_phi_chis(m_data, F_phi_lm, F_chi_lm, F_phi_ls, F_chi_ls)

In [ ]:
results

In [ ]:
m_data_radm = m_data * 2 * np.pi # [rad/m]

P_phi = np.log10(F_phi_ls['-7'])**2)
P_chi = np.log10(F_chi_ls['-7'])**2)

In [ ]:
def spectrum_model(m,GM, phi_noise, chi_noise, ray_id):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      
        
        zeta0 = 7.3 * np.sqrt(GM)
        rvs = tools.compute_n23_ray_variables(zeta0=zeta0, ray_ids=[ray_id])    

        # using first ray pair for now TODO
        F_phi, F_chi = tools.compute_F_phi_chi(rvs[ray_id][0], m)

    F_phi_noise = F_phi + phi_noise
    F_chi_noise = F_chi + chi_noise

    return F_phi_noise, F_chi_noise, F_phi, F_chi

In [ ]:
m = 2 * np.pi * np.logspace(-5, 0, 500)[None, :]  # [rad / m]
F_phi_model, F_chi_model, F_phi_nonoise, F_chi_nonoise = spectrum_model(m, 3, 1e0, 5e-1, '-7')

In [ ]:
def residuals(params, ray_id):
    
    GM, phi_noise, chi_noise = params

    P_phi = np.log10(F_phi_ls[ray_id])**2
    P_chi = np.log10(F_chi_ls[ray_id])**2
    
    F_phi_model, F_chi_model,_,_ = spectrum_model(m_data_radm, GM, phi_noise, chi_noise, ray_id)
    
    y_phi = np.log10(F_phi_model)
    f_phi = np.log10(F_phi_lm[ray_id])
    W_phi = np.diag(1/P_phi)

    y_chi = np.log10(F_chi_model)
    f_chi = np.log10(F_chi_lm[ray_id])
    W_chi = np.diag(1/P_chi)
    
    L_phi = (y_phi - f_phi).T @ W_phi @ (y_phi - f_phi)
    L_chi = (y_chi - f_chi).T @ W_chi @ (y_chi - f_chi)
    
    L = L_phi + L_chi
    
    return L

In [ ]:
residuals((2, 1e0, 2e-1), '-7')

In [ ]:
# Initial guess for [GM, phi_noise, chi_noise]
initial_params = np.array([1, 1e0, 2e-1])

# Minimize
result = optimize.minimize(residuals, initial_params, args=('-7',), method='Nelder-Mead')

In [ ]:
result

In [ ]:
F_phi_obs_noise['-7'], F_chi_obs_noise['-7']

In [ ]:
ray_id = '+6'

F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], ray_id)


sns.set_palette('tab20')

# F_phi modeled
plt.plot(m[0]/(2*np.pi), F_phi_model, c='C0', lw=1, linestyle='dashed')
plt.plot(m[0]/(2*np.pi), F_phi_model_nonoise, c='C0', lw=2)

# F_phi observed
plt.plot(m_data, F_phi_lm[ray_id], c="C1", marker="o", markersize=3)
plt.fill_between(
    m_data,
    F_phi_lm[ray_id] * F_phi_ls[ray_id],
    F_phi_lm[ray_id] / F_phi_ls[ray_id],
    alpha=0.3,
    color="C1",
    lw=0,
)

# F_chi modeled
plt.plot(m[0]/(2*np.pi), F_chi_model, c='C2', lw=1, linestyle='dashed')
plt.plot(m[0]/(2*np.pi), F_chi_model_nonoise, c='C2', lw=2)

# F_chi observed
plt.plot(m_data, F_chi_lm[ray_id], c="C3", marker="o", markersize=3)
plt.fill_between(
    m_data,
    F_chi_lm[ray_id] * F_chi_ls[ray_id],
    F_chi_lm[ray_id] / F_chi_ls[ray_id],
    alpha=0.3,
    color="C3",
    lw=0,
)

plt.yscale('log')
plt.xscale('log')

plt.title(f'{ray_id}: {results[ray_id].x[0]:0.3f} $\\times$ GM energy', loc='left')

## Plot Spectra

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams["text.usetex"] = True
sns.set_palette("tab20")

fig, axes = plt.subplots(2, 2, figsize=(6.67, 5))
axes = axes.flatten()

for k, ray_id in enumerate(["+7", "-7", "+6", "-6"]):
    
    F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], ray_id)

    # extend noise floor to the right
    nan_mask = np.isnan(F_phi_model)
    F_phi_model = np.hstack((F_phi_model[~nan_mask], F_phi_model[~nan_mask][-1]))
    F_chi_model = np.hstack((F_chi_model[~nan_mask], F_chi_model[~nan_mask][-1]))
    m_extend = np.concatenate((m[:,~nan_mask], np.array([[100]])), axis=1)
    
    plt.sca(axes[k])

    # WFT GMIW F_phi
    plt.plot(m[0] / (2 * np.pi), F_phi_model_nonoise, c="C0", lw=1.5)
    plt.plot(m_extend[0] / (2 * np.pi), F_phi_model, c="C0", lw=0.75, linestyle='dashed')
    
    # F_phi observed
    plt.plot(m_data, F_phi_lm[ray_id], c="C1", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_phi_lm[ray_id] * F_phi_ls[ray_id],
        F_phi_lm[ray_id] / F_phi_ls[ray_id],
        alpha=0.3,
        color="C1",
        lw=0,
    )

    # WFT GMIW F_chi
    plt.plot(m[0] / (2 * np.pi), F_chi_model_nonoise, c="C2", lw=1.5)
    plt.plot(m_extend[0] / (2 * np.pi), F_chi_model, c="C2", lw=0.75, linestyle='dashed')
    
    # F_chi observed
    plt.plot(m_data, F_chi_lm[ray_id], c="C3", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_chi_lm[ray_id] * F_chi_ls[ray_id],
        F_chi_lm[ray_id] / F_chi_ls[ray_id],
        alpha=0.3,
        color="C3",
        lw=0,
    )

    # Observational Noise Floor
    #plt.hlines([F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id]], 1e-2, 1e-0, colors=['C0', 'C2'])
    #print(F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id])
    
    plt.ylim([2e-4, 3e4])
    plt.xlim([2e-4, 4e-1])

    plt.xlabel("vertical wave number [1/m]")
    plt.ylabel("$F_{\\phi, \\chi}$ [$\\mathrm{rad} \\ m$]")

    plt.yscale("log")
    plt.xscale("log")

    plt.grid()

    plt.text(
        0.73,
        0.9,
        f"Ray ID: {ray_id}",
        transform=axes[k].transAxes,
        fontsize=9,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5),
    )
    
    plt.text(
        0.04,
        0.07,
        f"{results[ray_id].x[0]:0.1f} $\\times$ GM Energy\n$F_{{\\phi}}$ noise level = {results[ray_id].x[1]:0.2f} [rad m]\n$F_{{\\chi}}$ noise level = {results[ray_id].x[2]:0.2f} [rad m]",
        transform=axes[k].transAxes,
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5)
    )

plt.tight_layout()

# fig.savefig('../../figures/F_phi_chi_ducted.pdf', bbox_inches='tight')